# Traffic Density Analysis System

Chay cac cell theo thu tu tu tren xuong de khoi dong backend va detection trong cung notebook.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "detection").exists():
    PROJECT_ROOT = Path(r"d:/Code/TTCS/traffic-density-analysis-system")

os.chdir(PROJECT_ROOT)

for path in (PROJECT_ROOT, PROJECT_ROOT / "detection"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

ULTRALYTICS_CONFIG = PROJECT_ROOT / "detection" / "Ultralytics"
ULTRALYTICS_CONFIG.mkdir(parents=True, exist_ok=True)

os.environ["YOLO_CONFIG_DIR"] = str(ULTRALYTICS_CONFIG)
os.environ["YOLOV5_CONFIG_DIR"] = str(ULTRALYTICS_CONFIG)
os.environ["TRAFFIC_VIDEO_SOURCE"] = str(PROJECT_ROOT / "traffictrim.mp4")
os.environ["TRAFFIC_MODEL_PATH"] = str(PROJECT_ROOT / "detection" / "pro_models" / "best_final.pt")
os.environ["TRAFFIC_API_URL"] = "http://127.0.0.1:8000/detection"

VENV_PYTHON = PROJECT_ROOT / "venv" / "Scripts" / "python.exe"
BACKEND_LOG = PROJECT_ROOT / "backend_uvicorn.log"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("VENV_PYTHON:", VENV_PYTHON)
print("TRAFFIC_VIDEO_SOURCE:", os.environ["TRAFFIC_VIDEO_SOURCE"])
print("TRAFFIC_MODEL_PATH:", os.environ["TRAFFIC_MODEL_PATH"])
print("TRAFFIC_API_URL:", os.environ["TRAFFIC_API_URL"])


In [ ]:
required_paths = [
    PROJECT_ROOT / "traffictrim.mp4",
    PROJECT_ROOT / "detection" / "pro_models" / "best_final.pt",
    PROJECT_ROOT / "detection" / "configs_cameras" / "cam_01.json",
    PROJECT_ROOT / "backend" / "main.py",
]

missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_paths))

print("All required files are present.")


In [ ]:
import importlib

required_modules = [
    "fastapi",
    "uvicorn",
    "sqlalchemy",
    "requests",
    "cv2",
    "torch",
    "numpy",
    "scipy",
    "shapely",
    "supervision",
    "deep_sort_realtime",
    "dotenv",
]

missing_modules = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        missing_modules.append(f"{module_name}: {exc}")

if missing_modules:
    raise ImportError("Missing Python packages:\n" + "\n".join(missing_modules))

print("All required Python packages are available.")


In [ ]:
import socket
import subprocess
import time
import requests

def is_port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

def wait_for_backend(url: str, timeout: int = 30) -> None:
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            response = requests.get(url, timeout=1)
            if response.ok:
                return
        except requests.RequestException:
            time.sleep(1)
    raise TimeoutError(f"Backend did not become ready within {timeout} seconds: {url}")

if is_port_open("127.0.0.1", 8000):
    BACKEND_PROCESS = None
    print("Backend is already running on http://127.0.0.1:8000")
else:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT)
    BACKEND_LOG.write_text("", encoding="utf-8")
    backend_log_handle = open(BACKEND_LOG, "w", encoding="utf-8")
    BACKEND_PROCESS = subprocess.Popen(
        [str(VENV_PYTHON), "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
        cwd=str(PROJECT_ROOT),
        env=env,
        stdout=backend_log_handle,
        stderr=subprocess.STDOUT,
    )
    wait_for_backend("http://127.0.0.1:8000/openapi.json", timeout=30)
    print("Backend started on http://127.0.0.1:8000")
    print("Backend log:", BACKEND_LOG)


In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/openapi.json", timeout=3)
response.raise_for_status()

openapi = response.json()
print("Backend title:", openapi.get("info", {}).get("title"))
print("Available endpoints:", sorted(openapi.get("paths", {}).keys()))


In [ ]:
from detection.main import main

main()


In [ ]:
if 'BACKEND_PROCESS' in globals() and BACKEND_PROCESS is not None and BACKEND_PROCESS.poll() is None:
    BACKEND_PROCESS.terminate()
    BACKEND_PROCESS.wait(timeout=10)
    print("Backend stopped.")
else:
    print("No backend process started by this notebook.")
